# Race gait analysis on Google Colab

Upload a race video, pose-track your athlete with YOLO on a GPU, read off her
gait metrics (ground contact time, cadence, vertical bounce, head sway), and
turn them into a running-economy and finish-time estimate with the
`athlete_predictor` model.

**Before you start:** set a GPU runtime via *Runtime → Change runtime type →
Hardware accelerator: GPU (T4 is fine)*.

The broadcast cuts between cameras, so the video is split into shots; you point
at your athlete in each shot, and her gait is averaged over only the footage
where she's visible — off-screen time inherits that average.

In [ ]:
# 1. Confirm the GPU is attached.
!nvidia-smi -L

In [ ]:
# 2. Install the CV stack and get the project (clone, or pull if already cloned).
!pip -q install ultralytics opencv-python-headless
%cd /content
![ -d dance-pose-tracker ] && (cd dance-pose-tracker && git pull) || git clone -b claude/athletic-performance-predictor-gmhkll https://github.com/henosss/dance-pose-tracker.git
%cd /content/dance-pose-tracker

## 3. Upload your race video
Run the cell and pick the file. For a longer clip, mounting Google Drive
(`from google.colab import drive; drive.mount('/content/drive')`) and setting
`VIDEO` to the path there is faster than uploading.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO = next(iter(uploaded))
print('using', VIDEO)

In [ ]:
# 4. Who/what are we analysing?
ATHLETE       = 'Senayet Getachew'   # must match a name in the dataset
ATHLETE_HEIGHT_CM = 165              # used to calibrate pixels -> cm
EVENT         = '5000m'
GEAR          = 'super_spikes'
FATIGUE_ONSET = 0.6                  # None for a whole-race fault; 0.6 = late-race

# Speed knobs (see step 5):
MODEL      = 'yolov8s-pose.pt'   # 'n' fastest, 's' balanced, 'm'/'x' slower + better
IMGSZ      = 480                 # lower = faster (480 or 384 are good on a T4)
VID_STRIDE = 1                   # 2 = process every other frame (~2x faster)

## 5. Pose-track every runner, shot by shot
The heavy step. It now decodes the video **once** (cut detection happens inline)
and prints progress as it goes. The first run also downloads the model weights.

**If it's slow:** drop to `MODEL = 'yolov8n-pose.pt'`, lower `IMGSZ` to 384, or
set `VID_STRIDE = 2` in step 4. The next cell first does a quick 200-frame trial
so you can read the fps before committing to the whole clip — if the trial fps
looks fine, run the full cell below it.

In [ ]:
from athlete_predictor.video import extract_segments

# Quick trial: first ~200 frames, just to see the processing fps.
_ = extract_segments(VIDEO, model=MODEL, imgsz=IMGSZ, vid_stride=VID_STRIDE,
                     max_frames=200)
print('\nTrial done — if the fps above looks acceptable, run the next cell for the full clip.')

In [ ]:
# Full run over the whole clip.
shots = extract_segments(VIDEO, model=MODEL, imgsz=IMGSZ, vid_stride=VID_STRIDE)
print(f'{len(shots.shots)} camera shot(s) detected at {shots.fps:.0f} fps')

## 6. Find your athlete in each shot
Track IDs reset at every camera cut. The previews below have each runner's
**track ID drawn on**. Look through them, find your athlete, and note her ID in
each shot where she's clearly visible.

In [ ]:
import cv2, matplotlib.pyplot as plt
for i, img in enumerate(shots.previews):
    if img is None:
        continue
    plt.figure(figsize=(9, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'shot {i}  —  track IDs present: {shots.track_ids(i)}')
    plt.axis('off')
    plt.show()

In [ ]:
# 7. EDIT THIS: map each shot index to your athlete's track ID in that shot.
#    Skip shots where she isn't visible — they're simply not counted.
chosen = {
    0: 1,
    # 2: 3,
    # 5: 2,
}

In [ ]:
# 8. Save her keypoints and run the analysis.
from athlete_predictor.video import save_poses_json
from athlete_predictor.cli import main

save_poses_json('athlete_poses.json', shots, chosen,
                fps=shots.fps, athlete_height_cm=ATHLETE_HEIGHT_CM)

argv = ['analyze', '--poses', 'athlete_poses.json',
        '--athlete', ATHLETE, '--event', EVENT, '--gear', GEAR]
if FATIGUE_ONSET is not None:
    argv += ['--fatigue-onset', str(FATIGUE_ONSET)]
main(argv)

## 9. Bonus: equalized comparisons and what-ifs
The same model compares athletes on equal terms (peak age, same shoes) and runs
what-ifs. A few examples — edit freely:

In [ ]:
from athlete_predictor.cli import main
main(['compare', '--event', '5000m', '--gear', 'super_spikes',
      '--athletes', 'Freweyni Hailu', 'Senayet Getachew'])
print()
main(['predict', '--athlete', ATHLETE, '--event', EVENT, '--gear', GEAR,
      '--fix', 'head_wobble', '--fatigue-onset', '0.6'])